## 1. train-test 나누기

In [ ]:
import os, shutil, random, pathlib

# 경로 설정
IMG_ROOT = pathlib.Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251204")   # 이미지 최상위 경로
LBL_ROOT = pathlib.Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/obj_train_data")  # 라벨 최상위 경로 (txt가 들어있는 곳)
SPLIT_ROOT = pathlib.Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect") # 분할된 데이터셋을 저장할 새로운 최상위 경로

# 분할 비율 설정
splits = {"train":0.7, "val":0.2, "test":0.1}

# Check if SPLIT_ROOT is within IMG_ROOT or LBL_ROOT
if SPLIT_ROOT.is_relative_to(IMG_ROOT) or SPLIT_ROOT.is_relative_to(LBL_ROOT):
    print("Error: SPLIT_ROOT should not be inside IMG_ROOT or LBL_ROOT.")
    print(f"IMG_ROOT: {IMG_ROOT}")
    print(f"LBL_ROOT: {LBL_ROOT}")
    print(f"SPLIT_ROOT: {SPLIT_ROOT}")
else:
    # 라벨 파일 경로를 매핑하는 딕셔너리 생성
    # LBL_ROOT 내의 모든 txt 파일의 이름을 키로, 전체 경로를 값으로 저장
    label_map = {p.name: p for p in LBL_ROOT.rglob("*.txt")}
    print(f"Found {len(label_map)} label files in LBL_ROOT.")

    # 이미지와 라벨 쌍 수집
    # 사용자의 파일명 예시가 '.jpg'이므로, 'jpg' 파일을 검색하고 재귀적으로 모든 하위 폴더에서 찾도록 '**/*.jpg' 사용
    # 사용자 설명: "모든 이미지 수집 (하위폴더 포함)"
    all_imgs = list(IMG_ROOT.rglob("**/*.jpg")) # Collect all .jpg images recursively
    print(f"Found {len(all_imgs)} image files in IMG_ROOT (including subfolders).")

    # 라벨이 있는 이미지만 필터링
    labeled_data = []
    for img_path in all_imgs:
        label_name = img_path.with_suffix(".txt").name
        lbl_path = label_map.get(label_name) # 딕셔너리에서 라벨 경로 조회
        if lbl_path: # 라벨 경로가 존재하면
            labeled_data.append((img_path, lbl_path))

    # 이제 labeled_data는 (이미지 경로, 라벨 경로) 튜플의 리스트
    print(f"Found {len(labeled_data)} labeled image-label pairs. (Expected around 740 based on user input)")
    if len(labeled_data) != 740:
        print(f"Warning: Number of labeled pairs ({len(labeled_data)}) does not match user's expected 740. Please check your image/label paths.")

    random.seed(0); random.shuffle(labeled_data)

    n = len(labeled_data)
    cut1 = int(n*splits["train"])
    cut2 = int(n*(splits["train"]+splits["val"]))
    parts = {"train": labeled_data[:cut1], "val": labeled_data[cut1:cut2], "test": labeled_data[cut2:]}

    # 분할된 데이터셋 저장 폴더 생성
    os.makedirs(SPLIT_ROOT, exist_ok=True)

    for split, pairs in parts.items():
        if not pairs:
            print(f"Warning: No labeled data found for split: {split}")
            continue

        # 복사 대상 폴더 경로 생성 (예: /content/drive/.../dataset_split/train/images)
        # YOLOv8은 일반적으로 images와 labels 폴더 아래에 바로 파일을 저장합니다.
        img_dest_folder = SPLIT_ROOT / split / "images"
        lbl_dest_folder = SPLIT_ROOT / split / "labels"

        # 대상 폴더가 없으면 생성
        os.makedirs(img_dest_folder, exist_ok=True)
        os.makedirs(lbl_dest_folder, exist_ok=True)

        for img_path, lbl_path in pairs:
            # 파일 복사
            shutil.copy2(str(img_path), str(img_dest_folder / img_path.name))
            shutil.copy2(str(lbl_path), str(lbl_dest_folder / lbl_path.name))

    print("done")


Found 741 label files in LBL_ROOT.
Found 1259 image files in IMG_ROOT (including subfolders).
Found 740 labeled image-label pairs. (Expected around 740 based on user input)
done


## 2. yolov8 학습 시키기

In [1]:
### YOLOv8 Colab 학습 전체 코드
# 1. Ultralytics 설치
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00


In [ ]:


# 3. 데이터셋 경로 지정 (네 Drive 안 dataset 폴더)
DATASET_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect"
DATA_YAML = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect/data.yaml"
print("Using data.yaml:", DATA_YAML)

# 4. YOLO 불러오기
from ultralytics import YOLO

# Pretrained YOLOv8n (자동 다운로드)
model = YOLO("yolov8n.pt")  # 혹은 "yolov8n.yaml"으로 랜덤 초기화


Using data.yaml: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect/data.yaml
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# 5. 학습 시작
model.train(
    data=DATA_YAML,   # data.yaml 경로
    epochs=50,        # 반복 학습 횟수
    imgsz=640,        # 입력 이미지 크기
    batch=16,         # 배치 크기
    workers=2,        # 병렬 데이터로더 수 (Colab CPU 따라 조정)
    project="/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect/results",  # 결과 저장 폴더
    name="lettuce_yolov8n"                           # 결과 하위 이름
)

# 6. 학습 결과 확인 (mAP, loss 등)
results = model.val()
print(results)

# 7. 추론 테스트 (임의의 이미지 한 장)
TEST_IMG = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect/test/images/bed00_20251204_125027_cam0.jpg"
preds = model.predict(TEST_IMG, save=False, conf=0.25) # save=False로 설정하여 저장하지 않음

# 8. 결과 확인
from IPython.display import Image
# preds[0].plot() 함수를 사용하여 결과 이미지를 얻고, 이를 PIL Image로 변환 후 표시
from PIL import Image as PIL_Image
import numpy as np

result_img = preds[0].plot(img=np.array(PIL_Image.open(TEST_IMG)))
display(PIL_Image.fromarray(result_img))

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# 7. 추론 테스트 (임의의 이미지 한 장)
TEST_IMG = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/1. 원본/250811/20250811_001806.png"
preds = model.predict(TEST_IMG, save=False, conf=0.25) # save=False로 설정하여 저장하지 않음

# 8. 결과 확인
from IPython.display import Image
# preds[0].plot() 함수를 사용하여 결과 이미지를 얻고, 이를 PIL Image로 변환 후 표시
from PIL import Image as PIL_Image
import numpy as np

result_img = preds[0].plot(img=np.array(PIL_Image.open(TEST_IMG)))
display(PIL_Image.fromarray(result_img))

### 3. 이제 crop 으로 가자

In [2]:
"""
양상추 상단 베드 ROI를 YOLOv8 detection 모델로 자동 crop하는 스크립트
- 입력: 원본 이미지 루트 경로 (하위 폴더까지 모두 탐색)
- 출력: 동일한 하위 폴더 구조를 유지한 채 crop 이미지를 저장
- 특징:
  * ThreadPoolExecutor 를 이용한 병렬 처리
  * (옵션) 256x256 리사이즈 기능 포함
"""

import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
from ultralytics import YOLO

# ========================= 사용자 설정 ========================= #

# YOLO ROI detection 모델 경로 (예: 상단 베드 1클래스 모델)
MODEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect/results/lettuce_yolov8n3/weights/best.pt"  # <- 실제 모델 경로로 수정해서 사용

# 원본 이미지 루트 경로
INPUT_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212")

# crop된 이미지 저장 루트 경로
OUTPUT_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/2. ROI_BOX/251207-251212")

# 사용할 이미지 확장자
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# 병렬 처리 시 워커 개수
NUM_WORKERS = 8

# crop 후 256x256 리사이즈를 저장할지 여부 (True/False)
RESIZE_TO_256 = True


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:

# ============================================================ #

model = YOLO(MODEL_PATH)


def find_all_images(root: Path):
    """루트 경로 이하의 모든 이미지 파일 경로 리스트 반환"""
    image_paths = []
    for dirpath, _, filenames in os.walk(root):
        d = Path(dirpath)
        for fname in filenames:
            ext = Path(fname).suffix.lower()
            if ext in IMG_EXTS:
                image_paths.append(d / fname)
    return image_paths


def ensure_out_dir(img_path: Path) -> Path:
    """입력 이미지의 상대 경로를 기준으로 출력 디렉토리 생성 후, 저장 경로 반환"""
    rel_path = img_path.relative_to(INPUT_ROOT)  # 예: 2025-12-01/xxx.jpg
    out_dir = OUTPUT_ROOT / rel_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / rel_path.name
    return out_path


def crop_topbed_from_image(img_path: Path):
    """단일 이미지에서 상단 베드 ROI를 검출 후 crop하여 저장"""
    out_path = ensure_out_dir(img_path)

    # 이미 결과가 있으면 스킵 (원하면 주석 처리)
    if out_path.exists():
        return f"skip (exists): {img_path}"

    img = cv2.imread(str(img_path))
    if img is None:
        return f"fail (read error): {img_path}"

    # YOLO inference
    results = model.predict(img, verbose=False)
    if len(results) == 0:
        return f"fail (no result): {img_path}"

    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return f"fail (no box): {img_path}"

    # 여러 박스가 나와도 가장 큰 박스(면적 기준) 하나만 사용
    best_box = None
    best_area = -1

    for b in boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        w = max(0, x2 - x1)
        h = max(0, y2 - y1)
        area = w * h
        if area > best_area:
            best_area = area
            best_box = (x1, y1, x2, y2)

    if best_box is None or best_area <= 0:
        return f"fail (invalid box): {img_path}"

    x1, y1, x2, y2 = best_box
    h, w = img.shape[:2]

    # 이미지 경계를 벗어나지 않도록 클리핑
    x1 = max(0, min(x1, w - 1))
    x2 = max(0, min(x2, w))
    y1 = max(0, min(y1, h - 1))
    y2 = max(0, min(y2, h))

    if x2 <= x1 or y2 <= y1:
        return f"fail (clipped box invalid): {img_path}"

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        return f"fail (empty crop): {img_path}"

    # (옵션) 256x256 리사이즈
    if RESIZE_TO_256:
        crop = cv2.resize(crop, (256, 256), interpolation=cv2.INTER_AREA)

    cv2.imwrite(str(out_path), crop)
    return f"ok: {img_path} -> {out_path}"


def main():
    image_paths = find_all_images(INPUT_ROOT)
    print(f"발견한 이미지 개수: {len(image_paths)}")

    results = []
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        future_to_path = {ex.submit(crop_topbed_from_image, p): p for p in image_paths}
        for fut in as_completed(future_to_path):
            msg = fut.result()
            results.append(msg)
            # 진행상황 간단 출력
            if msg.startswith("ok"):
                print(msg)

    # 요약 출력
    n_ok = sum(1 for r in results if r.startswith("ok"))
    n_skip = sum(1 for r in results if r.startswith("skip"))
    n_fail = len(results) - n_ok - n_skip
    print("\n=== 요약 ===")
    print("성공:", n_ok)
    print("스킵(이미 존재):", n_skip)
    print("실패:", n_fail)


if __name__ == "__main__":
    main()


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
ok: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212/251208/bed19_20251208_174408_cam1.jpg -> /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/2. ROI_BOX/251207-251212/251208/bed19_20251208_174408_cam1.jpg
ok: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212/251208/bed00_20251208_174306_cam0.jpg -> /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/2. ROI_BOX/251207-251212/251208/bed00_20251208_174306_cam0.jpg
ok: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212/251208/bed01_20251208_174208_cam0.jpg -> /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이